### RAG Pipelines- Data Ingestion to vector DB pipline

In [112]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [113]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""

    all_documents = []
    pdf_dir = Path(pdf_directory)

    # find all PDF files (non-recursive)
    pdf_files = list(pdf_dir.glob("*.pdf"))
    print(f"Searching for PDF files in {pdf_dir}...")

    for pdf_file in pdf_files:
        print(f"\nProcessing file: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # add metadata
            for doc in documents:
                doc.metadata["source"] = pdf_file.name
                doc.metadata["file_path"] = str(pdf_file)

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages from {pdf_file.name}")

        except Exception as e:
            print(f"Error loading {pdf_file.name}: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents


# call the function
all_pdf_documents = process_all_pdfs("../data/pdf")
all_pdf_documents






Searching for PDF files in ..\data\pdf...

Processing file: 30-days-of-react-ebook-fullstackio.pdf
Loaded 304 pages from 30-days-of-react-ebook-fullstackio.pdf

Processing file: HolidayCalendar-2025.pdf
Loaded 1 pages from HolidayCalendar-2025.pdf

Processing file: Yatra Registration Letter - UTDB.pdf
Loaded 2 pages from Yatra Registration Letter - UTDB.pdf

Total documents loaded: 307


[Document(metadata={'producer': 'GPL Ghostscript 9.50', 'creator': 'Chromium', 'creationdate': '2020-02-27T16:02:56-06:00', 'moddate': '2020-02-27T16:02:56-06:00', 'author': 'Nate Murray', 'title': '30-days-of-react-book-cover.png', 'subject': '', 'keywords': '', 'source': '30-days-of-react-ebook-fullstackio.pdf', 'total_pages': 304, 'page': 0, 'page_label': '1', 'file_path': '..\\data\\pdf\\30-days-of-react-ebook-fullstackio.pdf'}, page_content=''),
 Document(metadata={'producer': 'GPL Ghostscript 9.50', 'creator': 'Chromium', 'creationdate': '2020-02-27T16:02:56-06:00', 'moddate': '2020-02-27T16:02:56-06:00', 'author': 'Nate Murray', 'title': '30-days-of-react-book-cover.png', 'subject': '', 'keywords': '', 'source': '30-days-of-react-ebook-fullstackio.pdf', 'total_pages': 304, 'page': 1, 'page_label': '2', 'file_path': '..\\data\\pdf\\30-days-of-react-ebook-fullstackio.pdf'}, page_content="Edit this page on Github (https://github.com/fullstackreact/30-days-of-react/blob/master/day-0

In [114]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

### Text splitting into chunks
def split_documents(documents, chunk_size=500, chunk_overlap=200):
    """Split documents into smaller chunks"""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split into {len(split_docs)} chunks.")

    if split_docs:
        print(f"Example chunk:\n{split_docs[0].page_content[:200]}...")
        print(f"Metadata:\n{split_docs[0].metadata}")

    return split_docs


In [115]:
chunks=split_documents(all_pdf_documents)
chunks

Split into 879 chunks.
Example chunk:
Edit this page on Github (https://github.com/fullstackreact/30-days-of-react/blob/master/day-01/post.md)
What is React?

Today, we're starting out at the beginning. Let's look at what
React is and wh...
Metadata:
{'producer': 'GPL Ghostscript 9.50', 'creator': 'Chromium', 'creationdate': '2020-02-27T16:02:56-06:00', 'moddate': '2020-02-27T16:02:56-06:00', 'author': 'Nate Murray', 'title': '30-days-of-react-book-cover.png', 'subject': '', 'keywords': '', 'source': '30-days-of-react-ebook-fullstackio.pdf', 'total_pages': 304, 'page': 1, 'page_label': '2', 'file_path': '..\\data\\pdf\\30-days-of-react-ebook-fullstackio.pdf'}


[Document(metadata={'producer': 'GPL Ghostscript 9.50', 'creator': 'Chromium', 'creationdate': '2020-02-27T16:02:56-06:00', 'moddate': '2020-02-27T16:02:56-06:00', 'author': 'Nate Murray', 'title': '30-days-of-react-book-cover.png', 'subject': '', 'keywords': '', 'source': '30-days-of-react-ebook-fullstackio.pdf', 'total_pages': 304, 'page': 1, 'page_label': '2', 'file_path': '..\\data\\pdf\\30-days-of-react-ebook-fullstackio.pdf'}, page_content="Edit this page on Github (https://github.com/fullstackreact/30-days-of-react/blob/master/day-01/post.md)\nWhat is React?\n\uf09b\nToday, we're starting out at the beginning. Let's look at what\nReact is and what makes it tick. We'll discuss why we want to\nuse it.\nOver the next 30 days, you'll get a good feel for the various parts of the \nReact\n(https:/ /facebook.github.io/react/) web framework and its ecosystem.\nEach day in our 30 day adventure will build upon the previous day's materials,"),
 Document(metadata={'producer': 'GPL Ghostscri

Embedding And vectorStoreDB

In [116]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List ,Dict ,Any ,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [117]:


class EmbeddingManager:
    """Handles document embeddings generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the EmbeddingManager with a specified SentenceTransformer model.

        Args:
            model_name (str): HuggingFace model name for SentenceTransformer
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading SentenceTransformer model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print("Model loaded successfully.")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts.

        Args:
            texts (List[str]): List of text strings to embed

        Returns:
            np.ndarray: Embeddings array
        """
        if self.model is None:
            raise ValueError("Model not loaded.")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)

        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
embedding_manager = EmbeddingManager()
embedding_manager
    

Loading SentenceTransformer model: all-MiniLM-L6-v2...
Model loaded successfully.


In [118]:
import os
import uuid
import numpy as np
import chromadb

from typing import List, Any


class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "../data/vector_store"
    ):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)

            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )

            print(f"Vector store initialized.")
            print(f"Collection name: {self.collection_name}")
            print(f"Existing documents: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain Document objects
            embeddings: Corresponding embeddings array
        """
        if len(documents) != len(embeddings):
            raise ValueError(
                "Number of documents must match number of embeddings"
            )

        print(f"Adding {len(documents)} documents to vector store...")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print("Documents added successfully.")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
vectorstore = VectorStore()

vectorstore
print(vectorstore.collection.count())

Vector store initialized.
Collection name: pdf_documents
Existing documents: 1766
1766


In [119]:
chunks

[Document(metadata={'producer': 'GPL Ghostscript 9.50', 'creator': 'Chromium', 'creationdate': '2020-02-27T16:02:56-06:00', 'moddate': '2020-02-27T16:02:56-06:00', 'author': 'Nate Murray', 'title': '30-days-of-react-book-cover.png', 'subject': '', 'keywords': '', 'source': '30-days-of-react-ebook-fullstackio.pdf', 'total_pages': 304, 'page': 1, 'page_label': '2', 'file_path': '..\\data\\pdf\\30-days-of-react-ebook-fullstackio.pdf'}, page_content="Edit this page on Github (https://github.com/fullstackreact/30-days-of-react/blob/master/day-01/post.md)\nWhat is React?\n\uf09b\nToday, we're starting out at the beginning. Let's look at what\nReact is and what makes it tick. We'll discuss why we want to\nuse it.\nOver the next 30 days, you'll get a good feel for the various parts of the \nReact\n(https:/ /facebook.github.io/react/) web framework and its ecosystem.\nEach day in our 30 day adventure will build upon the previous day's materials,"),
 Document(metadata={'producer': 'GPL Ghostscri

In [120]:
### convert the text to embeddings and store in vector db

texts=[doc.page_content for doc in chunks]
# texts

#generate embeddings
embeddings = embedding_manager.generate_embeddings(texts)

#store in vector db
vectorstore.add_documents(chunks, embeddings)


Generating embeddings for 879 texts...


Batches: 100%|██████████| 28/28 [00:16<00:00,  1.67it/s]


Generated embeddings with shape: (879, 384)
Adding 879 documents to vector store...
Documents added successfully.
Total documents in collection: 2645


Retriever Pipeline From VectoreStore

In [121]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [122]:
rag_retriever

In [154]:
rag_retriever.retrieve("jsx")

Retrieving documents for query: 'jsx'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 75.09it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_c4874655_111',
  'content': "previously, JSX is really just JavaScript executed by the browser. We can\nexecute JavaScript functions inside the JSX content as it will just get run by\nthe browser like the rest of our JavaScript.\nLet's move our activity item JSX inside of the function of the \nmap  function\nthat we'll run over for every item.\n40",
  'metadata': {'author': 'Nate Murray',
   'producer': 'GPL Ghostscript 9.50',
   'content_length': 314,
   'total_pages': 304,
   'page': 40,
   'title': '30-days-of-react-book-cover.png',
   'creationdate': '2020-02-27T16:02:56-06:00',
   'keywords': '',
   'file_path': '..\\data\\pdf\\30-days-of-react-ebook-fullstackio.pdf',
   'subject': '',
   'source': '30-days-of-react-ebook-fullstackio.pdf',
   'creator': 'Chromium',
   'page_label': '41',
   'doc_index': 111,
   'moddate': '2020-02-27T16:02:56-06:00'},
  'similarity_score': 0.2790260314941406,
  'distance': 0.7209739685058594,
  'rank': 1},
 {'id': 'doc_e6b8f3fb_111',


RAG Pipeline- VectorDB To LLM Output Generation 

In [151]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()
# print(os.getenv("GROQ_API_KEY"))

llm=ChatGroq(model="llama-3.3-70b-versatile", api_key=os.getenv("GROQ_API_KEY"),temperature=0.1,max_tokens=1024)

##2. Simple Rag function : retreive context +generate response
def rag_simple(query,retriever,llm,top_k=5):
  # retrieve relevant contexts
  results=retriever.retrieve(query,top_k=top_k)
  contexts="\n\n".join("\n\n".join([doc['content'] for doc in results])) if results else ""
  if not contexts:
      return "No relevant documents found."
  ## generate the answer using GROQ LLM.
  prompt = f"""
  You are a helpful AI assistant.
  Use ONLY the information from the contexts below to answer the question.
  If the answer is not present in the context, say "I don't know".

  {contexts}

  Question: {query}
  Answer:
  """
  # prompt+=f"\n\n{contexts}\n\nQuestion: {query}\nAnswer:"
  # response=llm.generate([prompt])
  response=llm.invoke([prompt.format(contexts=contexts,query=query  )])
  return response.content

In [152]:
answer = rag_simple(
    query="Hello World",
    retriever=rag_retriever,
    llm=llm,
    top_k=5
)

print(answer)

Retrieving documents for query: 'Hello World'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 94.11it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
No relevant documents found.


In [163]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Complex Components", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Complex Components'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 89.96it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
Answer: No relevant context found.
Sources: []
Confidence: 0.0
Context Preview: 


In [165]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("react", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'react'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 58.52it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
of the form and include each of these interface elements inside of it.
Importantly, each component in a React app abides by strict data
management principles. Complex, interactive user interfaces often involve
complex data and application state. The s

urface area of React is limited and
aimed at giving us the tools to be able to anticipate how our application will
look with a given set of circumstances. We dig into these principles later in
the course.

of the form and include each of these interface elements inside of it.
Importantly, each component in a React app abides by strict data
management principles. Complex, interactive user interfaces often involve
complex data and application state. The surface area of React is limited and
aimed at giving us the tools to be able to anticipate how our application will
look with a given set of circumstances. We dig into these principles later in
the course.

of the form and include each of these interface elements inside of it.
Importantly, each component in a React app abides by strict data
management principles. Complex, interactive user interfaces often involve
complex data and application state. The surface area of React is limited and
aimed at giving us the tools to be able to anticip